# Bootstrap Validation Methods

Bootstrap methods are particularly useful for:
- Small medical datasets
- Estimating confidence intervals
- Out-of-bag error estimation
- Bias-corrected performance estimates

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import sys
sys.path.append('..')
from trustcv.splitters.iid import BootstrapValidation

np.random.seed(42)
plt.style.use('seaborn-v0_8')

## 1. Basic Bootstrap Validation

Bootstrap resampling with replacement to estimate performance.

In [ ]:
# Generate small medical dataset
n_samples = 200  # Small dataset typical in medical research
n_features = 10

X = np.random.randn(n_samples, n_features)
y = (X[:, 0] + X[:, 1] + 0.5 * np.random.randn(n_samples) > 0).astype(int)

print(f"Dataset size: {n_samples} samples")
print(f"Class distribution: {np.bincount(y)}")
print(f"Class balance: {np.bincount(y) / len(y)}")

In [ ]:
# Standard bootstrap validation
bootstrap = BootstrapValidation(n_bootstraps=100, method='standard')
model = RandomForestClassifier(n_estimators=50, random_state=42)

bootstrap_scores = []
for bootstrap_idx, oob_idx in bootstrap.split(X, y):
    model.fit(X[bootstrap_idx], y[bootstrap_idx])
    score = model.score(X[oob_idx], y[oob_idx])
    bootstrap_scores.append(score)

print(f"Bootstrap scores: {len(bootstrap_scores)} iterations")
print(f"Mean accuracy: {np.mean(bootstrap_scores):.3f}")
print(f"95% CI: [{np.percentile(bootstrap_scores, 2.5):.3f}, {np.percentile(bootstrap_scores, 97.5):.3f}]")

## 2. .632 and .632+ Bootstrap

Bias-corrected bootstrap methods for better performance estimation.

In [ ]:
# .632 Bootstrap
bootstrap_632 = BootstrapValidation(n_bootstraps=100, method='632')
scores_632 = []

for bootstrap_idx, oob_idx in bootstrap_632.split(X, y):
    model.fit(X[bootstrap_idx], y[bootstrap_idx])
    
    # Training error
    train_score = model.score(X[bootstrap_idx], y[bootstrap_idx])
    
    # Out-of-bag error  
    oob_score = model.score(X[oob_idx], y[oob_idx]) if len(oob_idx) > 0 else train_score
    
    # .632 estimate
    score_632 = 0.368 * train_score + 0.632 * oob_score
    scores_632.append(score_632)

print(f".632 Bootstrap mean: {np.mean(scores_632):.3f}")
print(f".632 Bootstrap 95% CI: [{np.percentile(scores_632, 2.5):.3f}, {np.percentile(scores_632, 97.5):.3f}]")

In [ ]:
# Compare different bootstrap methods
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Distribution comparison
ax1.hist(bootstrap_scores, alpha=0.7, label='Standard Bootstrap', bins=20)
ax1.hist(scores_632, alpha=0.7, label='.632 Bootstrap', bins=20)
ax1.set_xlabel('Accuracy Score')
ax1.set_ylabel('Frequency')
ax1.set_title('Bootstrap Score Distributions')
ax1.legend()

# Box plot comparison
data_to_plot = [bootstrap_scores, scores_632]
labels = ['Standard', '.632']
ax2.boxplot(data_to_plot, labels=labels)
ax2.set_ylabel('Accuracy Score')
ax2.set_title('Bootstrap Method Comparison')

plt.tight_layout()
plt.show()

## Summary

Bootstrap methods are valuable for ML:

- **Standard Bootstrap**: Good for confidence intervals
- **.632 Bootstrap**: Reduces bias in performance estimates
- **.632+ Bootstrap**: Further bias correction for overfitted models

Particularly useful for small clinical datasets where traditional CV might be unstable.